# Test6 research — STU-Net on AWS cases (Test5 preprocess)

End-to-end research flow:

1. **Repo + setup** (Colab Drive or local/cluster)
2. **Download** a few RADCURE + HECKTOR cases from AWS
3. **Test5 preprocessing** — improved background, separate GTVp/GTVn, anatomy QC (0.50), canonical H&N dict
4. **Prepare STU-Net** — clone, weights, stage nnUNet CTs
5. **Predict + Dice** — STU-Net organs vs our GT (name-matched); tumor overlay for context only

| Goal | This notebook |
|------|----------------|
| AWS download | yes |
| Test5-style preprocess | yes (`CaseProcessor`) |
| STU-Net pretrained inference | yes (default **STU-Net-S**) |
| Tumor (GTVp/GTVn) prediction | **no** — not in pretrained classes |
| Organ Dice | yes — **name-matched** only |

Paper / code: [arXiv:2304.06716](https://arxiv.org/abs/2304.06716) · [uni-medical/STU-Net](https://github.com/uni-medical/STU-Net) · [`README.md`](README.md)


## 0. Colab setup — Drive, repo, package

**Skip this section on cluster** if `radcure-medical-imaging` is already installed and `cwd` is the repo.

**Colab tip:** after the install cell → **Runtime → Restart session** → remount Drive → run the import-only cell (skip pip).


In [ ]:
# Colab only — mount Drive. On cluster: skip / ignore ImportError.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("Not Colab — skip Drive mount")


In [ ]:
# Paths: prefer an existing clone on Drive; otherwise clone from GitHub.
# On cluster: set RADCURE_REPO / run from repo root.
import os
import subprocess
from pathlib import Path

IN_COLAB = Path("/content/drive").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
    DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
    GITHUB_REPO_URL = "https://github.com/xiscapericas/my_tailors_drawer.git"
    CLONE_DIR = Path("/content/my_tailors_drawer")
    REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")

    if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
        REPO_ROOT = DRIVE_REPO
        print("Using Drive repo:", REPO_ROOT)
    else:
        if not (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
            subprocess.check_call(
                ["git", "clone", "--depth", "1", GITHUB_REPO_URL, str(CLONE_DIR)]
            )
        REPO_ROOT = CLONE_DIR / REPO_SUBPATH
        print("Using cloned repo:", REPO_ROOT)
else:
    env_repo = os.environ.get("RADCURE_REPO")
    if env_repo:
        REPO_ROOT = Path(env_repo)
    else:
        REPO_ROOT = Path.cwd().resolve()
        for p in [REPO_ROOT, *REPO_ROOT.parents]:
            if (p / "setup.py").is_file() and (p / "image_processor").is_dir():
                REPO_ROOT = p
                break
    print("Using local/cluster repo:", REPO_ROOT)

assert (REPO_ROOT / "setup.py").is_file(), f"setup.py not found under {REPO_ROOT}"
assert (REPO_ROOT / "image_processor").is_dir()
os.chdir(REPO_ROOT)
print("cwd:", Path.cwd())


In [ ]:
# Install package deps (+ Totalsegmentator for Test5 preprocess).
# After this on Colab: Runtime → Restart session → remount Drive → import-only cell.
# On cluster with env already set up: you can skip or run lightly.

import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = Path("/content/drive").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
    DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
    CLONE_DIR = Path("/content/my_tailors_drawer")
    REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")
    if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
        REPO_ROOT = DRIVE_REPO
    elif (CLONE_DIR / REPO_SUBPATH / "setup.py").is_file():
        REPO_ROOT = CLONE_DIR / REPO_SUBPATH
    else:
        raise FileNotFoundError("Re-run Drive mount + clone cell first.")
else:
    REPO_ROOT = Path(os.environ.get("RADCURE_REPO", Path.cwd().resolve()))
    for p in [REPO_ROOT, *REPO_ROOT.parents]:
        if (p / "setup.py").is_file() and (p / "image_processor").is_dir():
            REPO_ROOT = p
            break

os.chdir(REPO_ROOT)
print("Installing from:", REPO_ROOT)

pkgs = [
    "numpy>=2.1,<2.3",
    "boto3",
    "python-dotenv",
    "blosc2>=2.5.0",
    "nibabel",
    "SimpleITK",
    "pydicom",
    "rt-utils",
    "matplotlib",
    "pandas",
    "scikit-image>=0.19.0,<0.26.0",
    "scipy",
    "opencv-python-headless",
    "tqdm",
    "p-tqdm",
    "seaborn",
    "gdown",
]

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade", *pkgs],
    cwd=str(REPO_ROOT),
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)],
    cwd=str(REPO_ROOT),
)
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "totalsegmentator", "numpy>=2.1,<2.3"],
    cwd=str(REPO_ROOT),
)

print("Install done from", REPO_ROOT)
if IN_COLAB:
    print(">>> Runtime → Restart session")
    print(">>> Then: remount Drive → run NEXT cell (skip this pip cell)")


In [ ]:
# Run AFTER Runtime → Restart session (Colab). Skip pip.
# Cluster: run once at start of session.

import os
import sys
from pathlib import Path

IN_COLAB = Path("/content/drive").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
    DRIVE_REPO = DRIVE_ROOT / "repos" / "radcure-medical-imaging"
    CLONE_DIR = Path("/content/my_tailors_drawer")
    REPO_SUBPATH = Path("psyduck-doing-phd/radcure-medical-imaging")
    if DRIVE_REPO.is_dir() and (DRIVE_REPO / "setup.py").is_file():
        REPO_ROOT = DRIVE_REPO
    else:
        REPO_ROOT = CLONE_DIR / REPO_SUBPATH
else:
    REPO_ROOT = Path(os.environ.get("RADCURE_REPO", Path.cwd().resolve()))
    for p in [REPO_ROOT, *REPO_ROOT.parents]:
        if (p / "setup.py").is_file() and (p / "image_processor").is_dir():
            REPO_ROOT = p
            break

assert (REPO_ROOT / "image_processor").is_dir(), f"Repo not found at {REPO_ROOT}"
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import image_processor

print("cwd:", Path.cwd())
print("numpy:", np.__version__)
print("image_processor OK:", image_processor.__file__)


## 1. Config, AWS credentials, case list

Test5 preprocess settings match `pipelines/test5`:

- `tumor_label_mode=separate`
- `background_mode=improved`
- `anatomy_qc_threshold=0.50`
- canonical H&N organ dictionary


In [ ]:
import os
import json
import shutil
import subprocess
import sys
import zipfile
from getpass import getpass
from pathlib import Path
from urllib.parse import urlparse

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import pandas as pd

# --- work roots ---
IN_COLAB = Path("/content/drive").is_dir() or bool(os.environ.get("COLAB_RELEASE_TAG"))
if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/phD/phD-Petia")
    TEST6_WORK_ROOT = Path(
        os.getenv("TEST6_WORK_ROOT", str(DRIVE_ROOT / "research_test6_stunet"))
    )
else:
    TEST6_WORK_ROOT = Path(
        os.getenv("TEST6_WORK_ROOT", "/media/HDD_8TB/xisca/work/research_test6_stunet")
    )

PREPROCESS_ROOT = TEST6_WORK_ROOT / "preprocess"
RADCURE_ROOT = PREPROCESS_ROOT / "TotalSegmentatorRetrain"  # CaseProcessor layout
HECKTOR_ROOT = PREPROCESS_ROOT / "hecktor"
HECKTOR_DOWNLOAD_DIR = PREPROCESS_ROOT / "hecktor_download"
ORGAN_DICT_PATH = TEST6_WORK_ROOT / "organ_dictionary_test6.json"
CANONICAL_DICT = (
    REPO_ROOT / "image_processor" / "resources" / "organ_dictionary_hn_canonical.json"
)
LABEL_ORDERS = REPO_ROOT / "research_notebooks" / "test6_stunet" / "label_orders.json"
STUNET_CLONE = TEST6_WORK_ROOT / "STU-Net"
NNUNET_PATH = Path(os.getenv("NNUNET_PATH", "/media/HDD_8TB/xisca/code/nnUNet"))

for p in (
    TEST6_WORK_ROOT,
    RADCURE_ROOT,
    HECKTOR_ROOT,
    HECKTOR_DOWNLOAD_DIR,
    *(TEST6_WORK_ROOT / d for d in ("weights", "inputs", "predictions", "dice", "figures", "results_folder")),
):
    p.mkdir(parents=True, exist_ok=True)

# --- secrets ---
try:
    from google.colab import userdata

    def _secret(name: str, default: str = "") -> str:
        try:
            return userdata.get(name)
        except Exception:
            return default
except ImportError:

    def _secret(name: str, default: str = "") -> str:
        return default


def ensure_env(key: str, prompt: str, secret: bool = False) -> str:
    val = os.environ.get(key) or _secret(key)
    if not val:
        val = getpass(prompt) if secret else input(prompt)
    os.environ[key] = val.strip()
    return os.environ[key]


ensure_env("AWS_ACCESS_KEY_ID", "AWS_ACCESS_KEY_ID: ", secret=True)
ensure_env("AWS_SECRET_ACCESS_KEY", "AWS_SECRET_ACCESS_KEY: ", secret=True)
os.environ.setdefault("AWS_REGION", _secret("AWS_REGION", "eu-west-1") or "eu-west-1")
os.environ["AWS_DEFAULT_REGION"] = os.environ["AWS_REGION"]

AWS_BUCKET_NAME = ensure_env(
    "AWS_BUCKET_NAME", "AWS_BUCKET_NAME (e.g. xisca-lab): ", secret=False
)
AWS_FOLDER = (
    os.environ.get("AWS_FOLDER")
    or _secret("AWS_FOLDER", "RADCURE/all_cases/")
    or "RADCURE/all_cases/"
)
os.environ["AWS_FOLDER"] = AWS_FOLDER

HECKTOR_S3_URI = (
    os.environ.get("HECKTOR_S3_URI")
    or _secret("HECKTOR_S3_URI", "s3://xisca-lab/HECKTOR/test1.zip")
    or "s3://xisca-lab/HECKTOR/test1.zip"
)
os.environ["HECKTOR_S3_URI"] = HECKTOR_S3_URI

# --- sample cases (small set for research) ---
RADCURE_CASE_IDS = ["RADCURE-0122", "RADCURE-0040"]
HECKTOR_CASE_IDS = ["HMR-012", "CHUM-023"]  # provisional centers; suffix fallback

# Test5 settings
ANATOMY_QC_THRESHOLD = float(os.getenv("TEST6_ANATOMY_QC", "0.50"))
STU_VARIANT = os.getenv("TEST6_STU_VARIANT", "small")  # small | base | large | huge
FAST_INFER = True

GDRIVE = {
    "small": "1HReH6dDrEuXgHPrsw7OrHSjvEUF3f4mv",
    "base": "1BHCp1Ort-OaVFwaZmvsG4qHiKiPeNb4h",
    "large": "1KA1eXWWf_xAoJg5KHYrxTmfiz7wxGhHS",
    "huge": "1Qrq7oGPJ7ileFHWOAxwpeWdaB6hySptU",
}
CHK_NAME = {
    "small": "small_ep4k",
    "base": "base_ep4k",
    "large": "large_ep4k",
    "huge": "huge_ep4k",
}
TRAINER = {
    "small": "STUNetTrainer_small",
    "base": "STUNetTrainer_base",
    "large": "STUNetTrainer_large",
    "huge": "STUNetTrainer_huge",
}

print("TEST6_WORK_ROOT:", TEST6_WORK_ROOT)
print("PREPROCESS_ROOT:", PREPROCESS_ROOT)
print("RADCURE:", RADCURE_CASE_IDS)
print("HECKTOR:", HECKTOR_CASE_IDS)
print("QC threshold:", ANATOMY_QC_THRESHOLD, "| STU variant:", STU_VARIANT)
print("AWS:", AWS_BUCKET_NAME, AWS_FOLDER)
print("HECKTOR S3:", HECKTOR_S3_URI)


## 2. Download cases from AWS

- **RADCURE:** zip per case via `AWSHandler` → unzip under `preprocess/TotalSegmentatorRetrain/`
- **HECKTOR:** S3 zip → extract matching case folders under `preprocess/hecktor/`


In [ ]:
from image_processor.io.aws_handler import AWSHandler
from image_processor.io.file_handler import FileHandler
from image_processor.conventions import get_hecktor_paths

aws = AWSHandler(
    bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    region_name=os.environ["AWS_REGION"],
)

# --- RADCURE ---
radcure_local = {}
for case_id in RADCURE_CASE_IDS:
    case_dir = RADCURE_ROOT / case_id
    zip_path = RADCURE_ROOT / f"{case_id}.zip"
    if not case_dir.is_dir():
        if not zip_path.is_file():
            print(f"Downloading {case_id} ...")
            aws.download_case(case_id, str(RADCURE_ROOT))
        print(f"Unzipping {case_id} ...")
        FileHandler.unzip_file(str(zip_path), str(case_dir))
    else:
        print(f"Already present: {case_dir}")
    radcure_local[case_id] = case_dir

print("RADCURE ready:", {k: str(v) for k, v in radcure_local.items()})


def resolve_hecktor_case_id(preferred_id: str, available_ids: list) -> str:
    if preferred_id in available_ids:
        return preferred_id
    num = preferred_id.split("-")[-1]
    matches = [a for a in available_ids if a.split("-")[-1] == num]
    if not matches:
        raise FileNotFoundError(
            f"No HECKTOR case for preferred {preferred_id} (suffix {num}). "
            f"Available sample: {available_ids[:15]}"
        )
    chosen = sorted(matches)[0]
    print(f"  {preferred_id} not found -> using {chosen} (suffix {num})")
    return chosen


def list_hecktor_case_ids_in_zip(zip_path: Path) -> list:
    ids = set()
    with zipfile.ZipFile(zip_path, "r") as zf:
        for n in zf.namelist():
            for part in Path(n).parts:
                if part.endswith(".nii.gz") or part in ("", ".", "__MACOSX"):
                    continue
                if "-" in part:
                    tail = part.split("-")[-1]
                    if tail.isdigit() and 1 <= len(tail) <= 4:
                        ids.add(part)
    return sorted(ids)


def hecktor_case_ready(cases_root: Path, case_id: str) -> bool:
    paths = get_hecktor_paths(str(cases_root / case_id), case_id)
    return os.path.isfile(paths["path_ct"]) and os.path.isfile(paths["path_mask"])


def download_s3_file(s3_uri: str, local_path: Path, region: str = "eu-west-1") -> Path:
    import boto3

    parsed = urlparse(s3_uri)
    bucket, key = parsed.netloc, parsed.path.lstrip("/")
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.is_file():
        print("Zip already on disk:", local_path)
        return local_path
    print(f"Downloading {s3_uri} -> {local_path}")
    boto3.client("s3", region_name=region).download_file(bucket, key, str(local_path))
    return local_path


def extract_hecktor_cases_from_zip(zip_path: Path, case_ids: list, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        names = zf.namelist()
        for case_id in case_ids:
            members = [
                n
                for n in names
                if f"/{case_id}/" in f"/{n}" or n.startswith(f"{case_id}/")
            ]
            if not members:
                raise FileNotFoundError(f"{case_id} not found inside {zip_path.name}")
            print(f"Extracting {case_id}: {len(members)} files")
            for m in members:
                zf.extract(m, dest_root)

    from pipelines.hecktor.test_pipeline import detect_hecktor_cases_root

    return Path(detect_hecktor_cases_root(str(dest_root)))


# --- HECKTOR ---
if all(hecktor_case_ready(HECKTOR_ROOT, c) for c in HECKTOR_CASE_IDS):
    HECKTOR_CASES_ROOT = HECKTOR_ROOT
    print("HECKTOR cases already under", HECKTOR_CASES_ROOT)
else:
    zip_name = Path(urlparse(HECKTOR_S3_URI).path).name or "hecktor.zip"
    zip_path = download_s3_file(
        HECKTOR_S3_URI, HECKTOR_DOWNLOAD_DIR / zip_name, os.environ["AWS_REGION"]
    )
    available = list_hecktor_case_ids_in_zip(zip_path)
    print(f"HECKTOR IDs in zip: n={len(available)}")
    HECKTOR_CASE_IDS = [resolve_hecktor_case_id(c, available) for c in HECKTOR_CASE_IDS]
    print("Resolved HECKTOR_CASE_IDS:", HECKTOR_CASE_IDS)
    unpack_parent = HECKTOR_DOWNLOAD_DIR / "unzipped_partial"
    found_root = extract_hecktor_cases_from_zip(zip_path, HECKTOR_CASE_IDS, unpack_parent)
    for case_id in HECKTOR_CASE_IDS:
        src = found_root / case_id
        dst = HECKTOR_ROOT / case_id
        if src.is_dir() and not dst.exists():
            shutil.copytree(src, dst)
        elif src.is_dir() and dst.exists():
            print("Already copied:", dst)
    HECKTOR_CASES_ROOT = HECKTOR_ROOT

for case_id in HECKTOR_CASE_IDS:
    assert hecktor_case_ready(HECKTOR_CASES_ROOT, case_id), f"Missing HECKTOR files for {case_id}"
print("All HECKTOR cases OK:", HECKTOR_CASE_IDS, "root=", HECKTOR_CASES_ROOT)


## 3. Test5 preprocessing (`CaseProcessor`)

Runs TotalSegmentator → improved background → combined mask with **separate GTVp/GTVn** → anatomy QC.

Outputs (nnUNet-style):

- RADCURE: `preprocess/TotalSegmentatorRetrain/{ID}/output/image|labels/`
- HECKTOR: `preprocess/hecktor/{ID}/output/image|labels/`

Skipped automatically if `output/image` + `output/labels` already exist (HECKTOR). For RADCURE, delete the case `output/` folder to force a re-run.


In [ ]:
from image_processor import (
    AnatomyQCRejected,
    CaseProcessor,
    HECKTOR,
    RADCURE,
    TUMOR_LABEL_MODE_SEPARATE,
)
from image_processor.conventions import get_hecktor_paths, get_nnunet_case_number
from image_processor.utils.image_processing import ImageProcessor

# Working copy of canonical dict (indices stay stable across cases)
if not ORGAN_DICT_PATH.is_file():
    shutil.copy2(CANONICAL_DICT, ORGAN_DICT_PATH)
print("Organ dict:", ORGAN_DICT_PATH)


def anatomy_qc_check(processor, case_id: str, convention: str, case_folder: Path) -> None:
    """Mirror Test5 QC (wired on relabel; process_case does not call it yet)."""
    if processor.anatomy_qc_threshold is None:
        return
    if convention == RADCURE:
        ct_path = case_folder / f"{case_id}.nii.gz"
        tumor_path = case_folder / f"{case_id}_tumor_mask_aligned.nii.gz"
    else:
        paths = get_hecktor_paths(str(case_folder), case_id)
        ct_path = Path(paths["path_ct"])
        tumor_path = Path(paths["path_mask"])
    if not ct_path.is_file() or not tumor_path.is_file():
        print(f"  QC skip (missing CT/tumor): {case_id}")
        return
    ct = nib.load(str(ct_path)).get_fdata().astype(np.float32)
    tumor = nib.load(str(tumor_path)).get_fdata().astype(np.int32)
    non_zero = ImageProcessor.get_non_zero_slices(tumor)
    z = tumor.shape[2]
    if not non_zero:
        slices = list(range(z))
    else:
        start = max(int(min(non_zero)) - processor.slice_expansion, 0)
        end = min(int(max(non_zero)) + processor.slice_expansion, z - 1)
        slices = list(range(start, end + 1))
    processor._maybe_reject_anatomy_qc(case_id, ct, tumor, slices)


processed = []  # list of {cohort, case_id, stem, image, label, status}


def collect_output_paths(case_id: str, convention: str, case_folder: Path) -> dict:
    num = get_nnunet_case_number(case_id, convention)
    stem = f"case_{num}"
    img = case_folder / "output" / "image" / f"{stem}_0000.nii.gz"
    lab_candidates = [
        case_folder / "output" / "labels" / f"{stem}.nii.gz",
        case_folder / "output" / "labels" / f"{stem}_0000.nii.gz",
    ]
    lab = next((p for p in lab_candidates if p.is_file()), lab_candidates[0])
    return {"stem": stem, "image": img, "label": lab}


# --- RADCURE ---
# CaseProcessor expects main_path; cases live under main_path/TotalSegmentatorRetrain/
rad_proc = CaseProcessor(
    main_path=str(PREPROCESS_ROOT),
    aws_bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    organ_dictionary_path=str(ORGAN_DICT_PATH),
    aws_region=os.environ["AWS_REGION"],
    convention=RADCURE,
    tumor_label_mode=TUMOR_LABEL_MODE_SEPARATE,
    background_mode="improved",
    anatomy_qc_threshold=ANATOMY_QC_THRESHOLD,
)

for case_id in RADCURE_CASE_IDS:
    case_folder = RADCURE_ROOT / case_id
    out_paths = collect_output_paths(case_id, RADCURE, case_folder)
    if out_paths["image"].is_file() and out_paths["label"].is_file():
        print(f"[RADCURE] skip (outputs exist): {case_id}")
        status = "skipped"
    else:
        try:
            result = rad_proc.process_case(case_id)
            anatomy_qc_check(rad_proc, case_id, RADCURE, case_folder)
            status = result.get("status", "success")
            print(f"[RADCURE] {case_id}: {status}")
            print("  image:", result.get("image_path"))
            print("  label:", result.get("label_path"))
            out_paths = {
                "stem": Path(result["image_path"]).name.replace("_0000.nii.gz", ""),
                "image": Path(result["image_path"]),
                "label": Path(result["label_path"]),
            }
        except AnatomyQCRejected as e:
            print(f"[RADCURE] QC rejected {case_id}: {e}")
            status = "qc_rejected"
            out_paths = collect_output_paths(case_id, RADCURE, case_folder)

    processed.append(
        {
            "cohort": "radcure",
            "case_id": case_id,
            "status": status,
            **{k: out_paths[k] for k in ("stem", "image", "label")},
        }
    )

# --- HECKTOR ---
hek_proc = CaseProcessor(
    main_path=str(PREPROCESS_ROOT),
    aws_bucket_name=AWS_BUCKET_NAME,
    aws_folder=AWS_FOLDER,
    organ_dictionary_path=str(ORGAN_DICT_PATH),
    aws_region=os.environ["AWS_REGION"],
    convention=HECKTOR,
    cases_root=str(HECKTOR_CASES_ROOT),
    tumor_label_mode=TUMOR_LABEL_MODE_SEPARATE,
    background_mode="improved",
    anatomy_qc_threshold=ANATOMY_QC_THRESHOLD,
)

for case_id in HECKTOR_CASE_IDS:
    case_folder = HECKTOR_CASES_ROOT / case_id
    out_paths = collect_output_paths(case_id, HECKTOR, case_folder)
    try:
        result = hek_proc.process_case(case_id)
        if result.get("status") != "skipped":
            anatomy_qc_check(hek_proc, case_id, HECKTOR, case_folder)
        status = result.get("status", "success")
        print(f"[HECKTOR] {case_id}: {status}")
        if result.get("image_path"):
            out_paths = {
                "stem": Path(result["image_path"]).name.replace("_0000.nii.gz", ""),
                "image": Path(result["image_path"]),
                "label": Path(result["label_path"]),
            }
    except AnatomyQCRejected as e:
        print(f"[HECKTOR] QC rejected {case_id}: {e}")
        status = "qc_rejected"

    processed.append(
        {
            "cohort": "hecktor",
            "case_id": case_id,
            "status": status,
            **{k: out_paths[k] for k in ("stem", "image", "label")},
        }
    )

samples = [
    s
    for s in processed
    if s["status"] != "qc_rejected"
    and Path(s["image"]).is_file()
    and Path(s["label"]).is_file()
]
print("\nReady for STU-Net:", len(samples), "cases")
for s in samples:
    print(f"  {s['cohort']} {s['case_id']} → {s['stem']}")
    print(f"    CT: {s['image']}")
    print(f"    GT: {s['label']}")


## 4. Prepare STU-Net (clone + weights + trainers)

Uses nnUNet v1-style `nnUNet_predict` with Task 101 + STU trainers.  
Default weights: **STU-Net-S** (`small_ep4k`).


In [ ]:
import gdown

if not STUNET_CLONE.is_dir():
    subprocess.check_call(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/uni-medical/STU-Net.git",
            str(STUNET_CLONE),
        ]
    )
else:
    print("STU-Net clone exists:", STUNET_CLONE)


def _copy_stunet_into_nnunet(nnunet_root: Path, stunet: Path) -> None:
    if not nnunet_root.is_dir():
        print(
            "NNUNET_PATH not found — skip copy; set NNUNET_PATH or install trainers manually:",
            nnunet_root,
        )
        return
    v1_train = nnunet_root / "nnunet" / "training" / "network_training"
    v1_arch = nnunet_root / "nnunet" / "network_architecture"
    src_train = stunet / "nnUNet-1.7.1" / "nnunet" / "training" / "network_training"
    src_arch = stunet / "nnUNet-1.7.1" / "nnunet" / "network_architecture"
    if not src_train.is_dir():
        src_train = stunet / "network_training"
    if not src_arch.is_dir():
        src_arch = stunet / "network_architecture"
    for src, dst in ((src_train, v1_train), (src_arch, v1_arch)):
        if src.is_dir() and dst.is_dir():
            for f in src.glob("STUNet*"):
                shutil.copy2(f, dst / f.name)
                print("copied", f.name, "→", dst)
        else:
            print("skip copy", src, "→", dst)


_copy_stunet_into_nnunet(NNUNET_PATH, STUNET_CLONE)

results_root = TEST6_WORK_ROOT / "results_folder"
os.environ["RESULTS_FOLDER"] = str(results_root)
os.environ["nnUNet_results"] = str(results_root)

task_dir = (
    results_root
    / "nnUNet"
    / "3d_fullres"
    / "Task101_TotalSegmentator"
    / f"{TRAINER[STU_VARIANT]}__nnUNetPlansv2.1"
)
fold_dir = task_dir / "fold_0"
fold_dir.mkdir(parents=True, exist_ok=True)

plan_src = (
    STUNET_CLONE / "plan_files" / f"{TRAINER[STU_VARIANT]}__nnUNetPlansv2.1" / "plans.pkl"
)
if not plan_src.is_file():
    candidates = list((STUNET_CLONE / "plan_files").rglob("plans.pkl"))
    plan_src = next((p for p in candidates if STU_VARIANT in str(p)), None)
assert plan_src and Path(plan_src).is_file(), f"plans.pkl not found under {STUNET_CLONE}/plan_files"
shutil.copy2(plan_src, task_dir / "plans.pkl")
print("plans:", task_dir / "plans.pkl")

model_path = fold_dir / f"{CHK_NAME[STU_VARIANT]}.model"
model_pkl = fold_dir / f"{CHK_NAME[STU_VARIANT]}.model.pkl"
if not model_path.is_file():
    print("Downloading STU-Net weights (Google Drive)…")
    out = TEST6_WORK_ROOT / "weights" / f"{CHK_NAME[STU_VARIANT]}_download"
    out.parent.mkdir(parents=True, exist_ok=True)
    gdown.download(id=GDRIVE[STU_VARIANT], output=str(out), quiet=False)
    try:
        with zipfile.ZipFile(out, "r") as zf:
            zf.extractall(TEST6_WORK_ROOT / "weights" / STU_VARIANT)
        found = list(
            (TEST6_WORK_ROOT / "weights" / STU_VARIANT).rglob(f"{CHK_NAME[STU_VARIANT]}.model")
        )
        assert found, "Downloaded zip but .model not found"
        shutil.copy2(found[0], model_path)
        pkl = found[0].with_suffix(found[0].suffix + ".pkl")
        if not pkl.is_file():
            pkl = Path(str(found[0]) + ".pkl")
        if pkl.is_file():
            shutil.copy2(pkl, model_pkl)
    except zipfile.BadZipFile:
        shutil.copy2(out, model_path)
else:
    print("Weights already present:", model_path)

print("Ready:", model_path.exists(), model_path)


## 5. Stage CTs + run STU-Net prediction


In [ ]:
input_dir = TEST6_WORK_ROOT / "inputs"
pred_dir = TEST6_WORK_ROOT / "predictions"
input_dir.mkdir(exist_ok=True)
pred_dir.mkdir(exist_ok=True)

for f in input_dir.glob("*.nii.gz"):
    f.unlink()

assert samples, "No samples ready — check preprocess / QC rejections above"

for s in samples:
    dst = input_dir / f"{s['stem']}_0000.nii.gz"
    shutil.copy2(s["image"], dst)
    print("staged", dst.name, "←", s["case_id"])

cmd = [
    "nnUNet_predict",
    "-i",
    str(input_dir),
    "-o",
    str(pred_dir),
    "-t",
    "101",
    "-m",
    "3d_fullres",
    "-f",
    "0",
    "-tr",
    TRAINER[STU_VARIANT],
    "-chk",
    CHK_NAME[STU_VARIANT],
]
if FAST_INFER:
    cmd += ["--mode", "fast", "--disable_tta"]

print("Running:\n ", " ".join(cmd))
env = os.environ.copy()
env["RESULTS_FOLDER"] = str(results_root)
if NNUNET_PATH.is_dir():
    env["PYTHONPATH"] = str(NNUNET_PATH) + os.pathsep + env.get("PYTHONPATH", "")

try:
    subprocess.check_call(cmd, env=env)
except FileNotFoundError:
    print(
        "nnUNet_predict not found on PATH.\n"
        "Activate the env where nnUNet v1 + STU trainers are installed.\n"
        "See research_notebooks/test6_stunet/README.md"
    )
    raise


## 6. Dice (name-matched organs)

STU-Net predicts **104 TotalSegmentator classes**. We score Dice only where names match our H&N dictionary (after light normalisation). **GTVp/GTVn are not predicted** by pretrained STU-Net.


In [ ]:
with open(LABEL_ORDERS) as f:
    stu_idx_to_name = {int(k): v for k, v in json.load(f).items()}

ours = {}
if ORGAN_DICT_PATH.is_file():
    with open(ORGAN_DICT_PATH) as f:
        ours = json.load(f)
elif CANONICAL_DICT.is_file():
    with open(CANONICAL_DICT) as f:
        ours = json.load(f)

ours_idx_to_name = {int(v): k for k, v in ours.items()}


def norm_name(n: str) -> str:
    return n.lower().replace("-", "_").replace(" ", "_")


stu_by_norm = {norm_name(v): k for k, v in stu_idx_to_name.items() if k > 0}
ours_by_norm = {
    norm_name(k): v
    for k, v in ours.items()
    if k not in ("background", "anatomical_region", "other-tissue", "GTVp", "GTVn")
}

matched = sorted(set(stu_by_norm) & set(ours_by_norm))
print(
    f"STU classes: {len(stu_idx_to_name)-1}  Our organs: {len(ours_by_norm)}  "
    f"Name matches: {len(matched)}"
)
print("Matched examples:", matched[:20])
print("Note: GTVp/GTVn are NOT in STU-Net pretrained classes.")


def dice_binary(a, b):
    a = a.astype(bool)
    b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    denom = a.sum() + b.sum()
    if denom == 0:
        return 1.0 if inter == 0 else 0.0
    return float(2 * inter / denom)


rows = []
for s in samples:
    pred_path = pred_dir / f"{s['stem']}.nii.gz"
    if not pred_path.is_file():
        alts = list(pred_dir.glob(f"{s['stem']}*.nii.gz"))
        pred_path = alts[0] if alts else None
    if pred_path is None or not Path(pred_path).is_file():
        print("Missing prediction for", s["stem"])
        continue

    pred = nib.load(str(pred_path)).get_fdata().astype(np.int32)
    gt = nib.load(str(s["label"])).get_fdata().astype(np.int32)
    if gt.shape != pred.shape:
        print(f"Shape mismatch {s['stem']}: gt {gt.shape} pred {pred.shape} — skip Dice")
        continue

    gtvp_idx = ours.get("GTVp")
    gtvn_idx = ours.get("GTVn")
    row = {
        "cohort": s["cohort"],
        "case_id": s["case_id"],
        "stem": s["stem"],
        "n_pred_labels": int(len(np.unique(pred)) - 1),
        "gtvp_voxels_gt": int(np.sum(gt == gtvp_idx)) if gtvp_idx is not None else None,
        "gtvn_voxels_gt": int(np.sum(gt == gtvn_idx)) if gtvn_idx is not None else None,
        "tumor_dice_stu": None,  # N/A — pretrained STU-Net has no tumor classes
    }

    dices = []
    for name in matched:
        si, oi = stu_by_norm[name], ours_by_norm[name]
        d = dice_binary(gt == oi, pred == si)
        row[f"dice_{name}"] = d
        dices.append(d)
    row["dice_matched_mean"] = float(np.mean(dices)) if dices else np.nan

    rows.append(row)
    print(
        f"{s['stem']} ({s['case_id']}): pred_labels={row['n_pred_labels']}  "
        f"matched_mean_dice={row['dice_matched_mean']:.4f}"
    )

df = pd.DataFrame(rows)
csv_path = TEST6_WORK_ROOT / "dice" / "stunet_sample_dice.csv"
df.to_csv(csv_path, index=False)
cols_show = [c for c in df.columns if not c.startswith("dice_") or c == "dice_matched_mean"]
try:
    display(df[cols_show])
except NameError:
    print(df[cols_show].to_string())
print("saved", csv_path)
if len(df) and "dice_matched_mean" in df.columns:
    print("Overall mean matched Dice:", float(df["dice_matched_mean"].mean()))


## 7. Visualisation — CT | STU-Net organs | our GT (+ tumor)


In [ ]:
def window_ct(vol, z, p=(1, 99)):
    sl = vol[:, :, z]
    lo, hi = np.percentile(sl, p)
    x = (sl - lo) / (hi - lo + 1e-8)
    return np.clip(x, 0, 1)


def overlay_labels(ax, base, lab, colors, alpha=0.45, highlight=None):
    ax.imshow(base.T, cmap="gray", origin="lower")
    rgba = np.zeros((*lab.shape, 4), dtype=np.float32)
    for idx, rgb in colors.items():
        if idx == 0:
            continue
        rgba[lab == idx] = (*rgb, alpha)
    if highlight:
        for idx, rgb in highlight.items():
            rgba[lab == idx] = (*rgb, 0.85)
    ax.imshow(np.transpose(rgba, (1, 0, 2)), origin="lower")
    ax.axis("off")


rng = np.random.default_rng(0)
stu_colors = {}
for i in range(1, 105):
    rgb = rng.random(3)
    if rgb[0] > 0.7 and rgb[1] < 0.45:
        rgb[0] = 0.2
    stu_colors[i] = tuple(rgb.tolist())

gt_colors = {i: (0.3, 0.7, 0.9) for i in range(1, 100)}
tumor_hi = {}
if ours.get("GTVp") is not None:
    tumor_hi[ours["GTVp"]] = (1.0, 0.0, 0.0)
if ours.get("GTVn") is not None:
    tumor_hi[ours["GTVn"]] = (1.0, 0.41, 0.71)

for s in samples:
    pred_path = pred_dir / f"{s['stem']}.nii.gz"
    if not pred_path.is_file():
        alts = list(pred_dir.glob(f"{s['stem']}*.nii.gz"))
        if not alts:
            continue
        pred_path = alts[0]
    ct = nib.load(str(s["image"])).get_fdata().astype(np.float32)
    pred = nib.load(str(pred_path)).get_fdata().astype(np.int32)
    gt = nib.load(str(s["label"])).get_fdata().astype(np.int32)

    z = ct.shape[2] // 2
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    base = window_ct(ct, z)
    axes[0].imshow(base.T, cmap="gray", origin="lower")
    axes[0].set_title(f"CT  z={z}")
    axes[0].axis("off")
    overlay_labels(axes[1], base, pred[:, :, z], stu_colors)
    axes[1].set_title("STU-Net organs")
    overlay_labels(axes[2], base, gt[:, :, z], gt_colors, highlight=tumor_hi)
    axes[2].set_title("Our GT (+ tumor)")
    fig.suptitle(f"{s['cohort']} | {s['case_id']} | {s['stem']} | STU-Net-{STU_VARIANT}")
    out = TEST6_WORK_ROOT / "figures" / f"{s['stem']}_stunet_explore.png"
    fig.savefig(out, dpi=120, bbox_inches="tight")
    plt.show()
    print("saved", out)


## 8. Takeaways / next

- This notebook downloads cases from AWS, applies **Test5 preprocessing**, then runs **pretrained STU-Net** organ inference.
- Pretrained STU-Net is an **organ** foundation model (104 TS classes), not a H&N tumor model — tumor Dice needs fine-tuning.
- Useful next steps:
  1. More samples / STU-Net-B if S looks promising on matched organs
  2. Map STU-Net organs → our H&N set more carefully (synonyms)
  3. Fine-tune on Dataset650 with GTVp/GTVn for a true Test6 vs Test4/5

When ready for a full experiment, promote into `pipelines/` + `experiments/registry.yaml` (`status: running`).
